"""
Trip-based pathways construction from GTFS and OSM graph.
Generates walking connections between trip starts/ends that are near other trips' stops.
This handles bidirectional routes properly by working with individual trips.
Output columns:
- start_trip_id, end_trip_id, start_stop_id, end_stop_id, walking_distance_m, walking_path_nodes
"""


In [1]:
import os
import math
import pandas as pd
import numpy as np
import osmnx as ox
import networkx as nx
from pathlib import Path

# Configure paths
GTFS_DIR = Path(r"C:/Users/ahmed/Downloads/gtfsAlex")
GRAPH_XML = Path(r"C:/Users/ahmed/Documents/grad/draft1/labeled.osm")

# Load GTFS
routes = pd.read_csv(GTFS_DIR / "routes.txt")
stops = pd.read_csv(GTFS_DIR / "stops.txt")
trips = pd.read_csv(GTFS_DIR / "trips.txt")
stop_times = pd.read_csv(GTFS_DIR / "stop_times.txt")
shapes = pd.read_csv(GTFS_DIR / "shapes.txt")

# Ensure correct dtypes
stop_times["stop_sequence"] = pd.to_numeric(stop_times["stop_sequence"], errors="coerce")

print(f"Loaded routes={len(routes)}, stops={len(stops)}, trips={len(trips)}, stop_times={len(stop_times)}")

# Load OSM graph (walkable)
g = ox.graph_from_xml(filepath=str(GRAPH_XML), bidirectional=True)
# Simplify graph for routing consistency
g = ox.convert.to_undirected(g)
print(f"Graph loaded with {g.number_of_nodes()} nodes, {g.number_of_edges()} edges")


Loaded routes=104, stops=441, trips=192, stop_times=2547
Graph loaded with 45784 nodes, 65854 edges


In [2]:
# Build trip -> ordered stops, and canonical start/end stop per trip
# We assume stop_times stop_sequence ascending defines direction along a trip.

# Filter trips that have stop_times data
trips_with_stops = trips[trips['trip_id'].isin(stop_times['trip_id'])]
print(f"Trips with stop_times data: {len(trips_with_stops)}")

# Build ordered stops for each trip
ordered_stops = (
    stop_times
    .sort_values(['trip_id','stop_sequence'])
    .merge(trips_with_stops[['trip_id','route_id']], on='trip_id', how='left')
)

# Pick start and end stop_id per trip
start_stop_per_trip = ordered_stops.groupby('trip_id').first()['stop_id']
end_stop_per_trip = ordered_stops.groupby('trip_id').last()['stop_id']

print(f"Trips with start/end stops: {len(start_stop_per_trip)}")

# Create convenience dicts
trip_to_start_stop = start_stop_per_trip.to_dict()
trip_to_end_stop = end_stop_per_trip.to_dict()

# Also collect all trip->set(stops) for proximity to the whole trip corridor
trip_to_all_stops = (
    ordered_stops.groupby('trip_id')['stop_id'].apply(lambda s: set(s.values)).to_dict()
)

# Create trip->route mapping for reference
trip_to_route = trips.set_index('trip_id')['route_id'].to_dict()

# Create trip->shape_id mapping for visualization
trip_to_shape = trips.set_index('trip_id')['shape_id'].to_dict()

# Build shape_id -> list of (lat, lon) points for visualization
shapes = pd.read_csv(GTFS_DIR / "shapes.txt")
shape_to_points = {}
for shape_id, group in shapes.groupby('shape_id'):
    points = group.sort_values('shape_pt_sequence')[['shape_pt_lat', 'shape_pt_lon']].values.tolist()
    shape_to_points[shape_id] = points

print(f"Built {len(shape_to_points)} shape geometries for visualization")


Trips with stop_times data: 192
Trips with start/end stops: 192
Built 192 shape geometries for visualization


In [3]:
# Map each GTFS stop to nearest graph node
# Prepare stop_id -> (lat, lon) -> nearest node
stop_coords = stops.set_index('stop_id')[['stop_lat','stop_lon']]

# Vectorized nearest nodes using OSMnx
xs = stop_coords['stop_lon'].values
ys = stop_coords['stop_lat'].values
nearest_nodes = ox.distance.nearest_nodes(g, X=xs, Y=ys)

stop_to_node = {stop_id: node for stop_id, node in zip(stop_coords.index.values, nearest_nodes)}
print(f"Mapped {len(stop_to_node)} stops to nearest graph nodes")

# Convenience: node positions
node_x = nx.get_node_attributes(g, 'x')
node_y = nx.get_node_attributes(g, 'y')


Mapped 441 stops to nearest graph nodes


In [8]:
# Compute proximity candidates:
# For each trip t1, find trips t2 whose start node is near any stop node of t1,
# and trips t3 whose end node is near any stop node of t1.
# We'll use a radius (meters) on great-circle distance to get candidates; exact walking
# shortest path will be computed in the next step.
# IMPORTANT: We ignore t1's own first stop when considering proximity to other trip starts.

from math import radians, sin, cos, asin, sqrt

def haversine_m(lat1, lon1, lat2, lon2):
    # returns meters
    R = 6371000.0
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    c = 2 * asin(sqrt(a))
    return R * c

# Build stop_id -> (lat, lon)
stop_lat = stops.set_index('stop_id')['stop_lat'].to_dict()
stop_lon = stops.set_index('stop_id')['stop_lon'].to_dict()

# Precompute start/end node for every trip
trip_to_start_node = {t: stop_to_node[s] for t, s in trip_to_start_stop.items() if s in stop_to_node}
trip_to_end_node = {t: stop_to_node[s] for t, s in trip_to_end_stop.items() if s in stop_to_node}

# Parameters
NEAR_RADIUS_M = 300  # coarse proximity filter in meters

# Build a fast lookup of trip t -> list of nodes for all its stops
trip_to_nodes = {t: [stop_to_node[s] for s in stops_set if s in stop_to_node]
                  for t, stops_set in trip_to_all_stops.items()}

# For reverse lookup: node -> (lat, lon)
node_latlon = {n: (node_y.get(n), node_x.get(n)) for n in g.nodes()}

# Candidate list: (start_trip_id, end_trip_id, start_node, end_node)
# Two types:
# 1) t2 start near t1 corridor: (t2 -> t1)
# 2) t3 end near t1 corridor: (t3 -> t1)
# Note: We ignore t1's own first stop when considering proximity to other trip starts.
proximity_candidates = []

trip_ids = list(trip_to_all_stops.keys())


for t1 in trip_ids:
    t1_nodes = trip_to_nodes.get(t1, [])
    if not t1_nodes:
        continue

    # compute t1's start node (to be excluded when checking other starts)
    t1_start_stop = trip_to_start_stop.get(t1)
    t1_start_node = stop_to_node.get(t1_start_stop) if t1_start_stop in stop_to_node else None

    # 1) t2 starts near t1
    for t2, t2_start_node in trip_to_start_node.items():
        if t2 == t1:
            continue
        # skip if the closest t1 node would be exactly its own start node
        lat2, lon2 = node_latlon.get(t2_start_node, (None, None))
        if lat2 is None:
            continue
        # coarse: check nearest t1 node by haversine radius
        near = False
        closest_t1_node = None
        closest_dist = float('inf')
        for n in t1_nodes:
            if t1_start_node is not None and n == t1_start_node:
                # ignore t1's start node when comparing to other starts
                continue
            lat1, lon1 = node_latlon.get(n, (None, None))
            if lat1 is None:
                continue
            d = haversine_m(lat1, lon1, lat2, lon2)
            if d < closest_dist:
                closest_dist = d
                closest_t1_node = n
            if d <= NEAR_RADIUS_M:
                near = True
                break
        if near and closest_t1_node is not None:
            # Corrected: Creates pathway from t1 -> t2
            proximity_candidates.append((t1, t2, closest_t1_node, t2_start_node))

    # 2) t3 ends near t1 (unchanged - no exclusion needed)
    for t3, t3_end_node in trip_to_end_node.items():
        if t3 == t1:
            continue
        lat3, lon3 = node_latlon.get(t3_end_node, (None, None))
        if lat3 is None:
            continue
        near = False
        closest_t1_node = None
        closest_dist = float('inf')
        for n in t1_nodes:
            lat1, lon1 = node_latlon.get(n, (None, None))
            if lat1 is None:
                continue
            d = haversine_m(lat1, lon1, lat3, lon3)
            if d < closest_dist:
                closest_dist = d
                closest_t1_node = n
            if d <= NEAR_RADIUS_M:
                near = True
                break
        if near:
            proximity_candidates.append((t3, t1, t3_end_node, closest_t1_node))

print(f"Proximity candidates: {len(proximity_candidates)}")


Proximity candidates: 6504


In [9]:
# Shortest walking paths for candidates and assemble DataFrame
# We find the closest actual stop node on t1 to the candidate endpoint node as the "end_stop_id".
# Path cost uses edge length in meters if available, else fallback to 1.

# Build reverse node->stop_id index to resolve end_stop_id
node_to_stop_ids = {}
for sid, n in stop_to_node.items():
    node_to_stop_ids.setdefault(n, []).append(sid)

# Helper to pick the nearest t1 stop node to a given node by straight distance
from math import inf

def nearest_node_in_set(target_node, node_set):
    ty, tx = node_y.get(target_node), node_x.get(target_node)
    if ty is None:
        return None
    best_node, best_d = None, inf
    for n in node_set:
        ny, nx_ = node_y.get(n), node_x.get(n)
        if ny is None:
            continue
        d = haversine_m(ty, tx, ny, nx_)
        if d < best_d:
            best_d = d
            best_node = n
    return best_node

# Ensure we have edge lengths (meters)
if not nx.get_edge_attributes(g, 'length'):
    g = ox.distance.add_edge_lengths(g)

# 1. Prerequisites: Lookup Dictionaries
# Map Trip -> Set of Stops (for validity check)
trip_to_stop_set = stop_times.groupby('trip_id')['stop_id'].apply(set).to_dict()

# Map Trip -> Route ID (Needed to populate the route_id columns)
trip_to_route = trips.set_index('trip_id')['route_id'].to_dict()

records = []
failed = 0

print(f"Processing {len(proximity_candidates)} proximity candidates...")

for start_trip_id, end_trip_id, start_node, coarse_end_node in proximity_candidates:
    
    # [1] Refine end node: find the nearest node used by the specific end_trip
    # Get all nodes associated with the end trip
    target_trip_nodes = [stop_to_node[s] for s in trip_to_stop_set.get(end_trip_id, []) if s in stop_to_node]
    
    if not target_trip_nodes:
        continue

    end_node = nearest_node_in_set(coarse_end_node, target_trip_nodes)
    if end_node is None:
        continue

    try:
        # [2] Calculate walking path
        path = nx.shortest_path(g, source=start_node, target=end_node, weight='length')
        
        # Compute distance
        dist = 0.0
        for u, v in zip(path[:-1], path[1:]):
            data = g.get_edge_data(u, v)
            if not data: continue
            ed = next(iter(data.values()))
            dist += float(ed.get('length', 1.0))

        # [3] Resolve Stop IDs
        # Start stop: (You might need to refine this if you want the specific stop at start_node)
        # For now, we take the stop from the trip that matches the start_node location if possible
        start_stop_id = None
        candidates_start = node_to_stop_ids.get(start_node, [])
        valid_stops_start = trip_to_stop_set.get(start_trip_id, set())
        for candidate in candidates_start:
            if candidate in valid_stops_start:
                start_stop_id = candidate
                break
        
        # End stop: (The fix we discussed)
        end_stop_id = None
        candidates_end = node_to_stop_ids.get(end_node, [])
        valid_stops_end = trip_to_stop_set.get(end_trip_id, set())
        
        for candidate in candidates_end:
            if candidate in valid_stops_end:
                end_stop_id = candidate
                break
        
        if end_stop_id is None: # Should not happen given logic above, but safe to skip
            continue

        # [4] Retrieve Route IDs (FIX FOR YOUR ERROR)
        start_route_id = trip_to_route.get(start_trip_id)
        end_route_id = trip_to_route.get(end_trip_id)

        records.append({
            'start_trip_id': start_trip_id,
            'end_trip_id': end_trip_id,
            'start_route_id': start_route_id, # <--- Added this
            'end_route_id': end_route_id,     # <--- Added this
            'start_stop_id': start_stop_id,
            'end_stop_id': end_stop_id,
            'walking_distance_m': dist,
            'walking_path_nodes': path,
        })

    except Exception as e:
        failed += 1

print(f"Built {len(records)} pathways. Failed: {failed}")

# Re-create the DataFrame
trip_pathways_df = pd.DataFrame.from_records(records)
trip_pathways_df.head()

Processing 6504 proximity candidates...
Built 6504 pathways. Failed: 0


,start_trip_id,end_trip_id,start_route_id,end_route_id,start_stop_id,end_stop_id,walking_distance_m,walking_path_nodes
0,-Q2gVX9GgVSEtVr-yv6Nf-07:00:00,3SynmEGmJSDoBYRTRgEkj-07:00:00,lgA0Qn219kk386RQHCTQ7,rJNlVBeJYLnin9iUCfQ8_,310,310,0.000000,[5275557217]
1,-Q2gVX9GgVSEtVr-yv6Nf-07:00:00,mOAyR9hC46n6ORhDBT2l7-07:00:00,lgA0Qn219kk386RQHCTQ7,lgA0Qn219kk386RQHCTQ7,443,443,0.000000,[6311195586]
2,PAh8O-96IhPU2mfK1XGjm-07:00:00,-Q2gVX9GgVSEtVr-yv6Nf-07:00:00,xg6X_6yI3-x6agQaIKy0r,lgA0Qn219kk386RQHCTQ7,308,309,39.742259,"[1128480896, 1128480646, 1128479568]"
3,_npHlyCCY7o0R20RyqvT8-07:00:00,-Q2gVX9GgVSEtVr-yv6Nf-07:00:00,rJNlVBeJYLnin9iUCfQ8_,lgA0Qn219kk386RQHCTQ7,311,310,7073.199280,"[8011331170, 7928660351, 7928645036, 792864503..."
4,-vGMBy-ffWWJVczaRlv5z-07:00:00,8wpT6YlGJmfsskCDg8AJT-07:00:00,d1tk5YD606wPnGF4CLm4i,RSdPdtwiGzlvedMSUpA36,321,322,139.949724,"[2712663245, 6948570678, 6953697312, 695369731..."


In [10]:
# Save to CSV and preview
output_csv = Path("trip_pathways.csv")
trip_pathways_df.to_csv(output_csv, index=False)
print(f"Saved trip pathways to {output_csv.resolve()}")
trip_pathways_df.head(5)


Saved trip pathways to C:\Users\ahmed\Documents\grad\draft1\trip_pathways.csv


,start_trip_id,end_trip_id,start_route_id,end_route_id,start_stop_id,end_stop_id,walking_distance_m,walking_path_nodes
0,-Q2gVX9GgVSEtVr-yv6Nf-07:00:00,3SynmEGmJSDoBYRTRgEkj-07:00:00,lgA0Qn219kk386RQHCTQ7,rJNlVBeJYLnin9iUCfQ8_,310,310,0.000000,[5275557217]
1,-Q2gVX9GgVSEtVr-yv6Nf-07:00:00,mOAyR9hC46n6ORhDBT2l7-07:00:00,lgA0Qn219kk386RQHCTQ7,lgA0Qn219kk386RQHCTQ7,443,443,0.000000,[6311195586]
2,PAh8O-96IhPU2mfK1XGjm-07:00:00,-Q2gVX9GgVSEtVr-yv6Nf-07:00:00,xg6X_6yI3-x6agQaIKy0r,lgA0Qn219kk386RQHCTQ7,308,309,39.742259,"[1128480896, 1128480646, 1128479568]"
3,_npHlyCCY7o0R20RyqvT8-07:00:00,-Q2gVX9GgVSEtVr-yv6Nf-07:00:00,rJNlVBeJYLnin9iUCfQ8_,lgA0Qn219kk386RQHCTQ7,311,310,7073.199280,"[8011331170, 7928660351, 7928645036, 792864503..."
4,-vGMBy-ffWWJVczaRlv5z-07:00:00,8wpT6YlGJmfsskCDg8AJT-07:00:00,d1tk5YD606wPnGF4CLm4i,RSdPdtwiGzlvedMSUpA36,321,322,139.949724,"[2712663245, 6948570678, 6953697312, 695369731..."


In [11]:
# Add route names for better readability
route_id_to_name = routes.set_index('route_id')['route_long_name'].to_dict()

trip_pathways_df['start_route_name'] = trip_pathways_df['start_route_id'].map(route_id_to_name)
trip_pathways_df['end_route_name'] = trip_pathways_df['end_route_id'].map(route_id_to_name)

# Show some examples
trip_pathways_df[['start_trip_id', 'start_route_name', 'end_trip_id', 'end_route_name', 
                  'start_stop_id', 'end_stop_id', 'walking_distance_m']].head(3)


,start_trip_id,start_route_name,end_trip_id,end_route_name,start_stop_id,end_stop_id,walking_distance_m
0,-Q2gVX9GgVSEtVr-yv6Nf-07:00:00,Bakus - El-Awayed,3SynmEGmJSDoBYRTRgEkj-07:00:00,El-Mansheya - Hagar Al-Nawateyah (Namos Bridge),310,310,0.000000
1,-Q2gVX9GgVSEtVr-yv6Nf-07:00:00,Bakus - El-Awayed,mOAyR9hC46n6ORhDBT2l7-07:00:00,Bakus - El-Awayed,443,443,0.000000
2,PAh8O-96IhPU2mfK1XGjm-07:00:00,Abo Soliman - El-Mansheya,-Q2gVX9GgVSEtVr-yv6Nf-07:00:00,Bakus - El-Awayed,308,309,39.742259


In [12]:
# Visualization: plot two trips and the walking path between them
import folium

def _path_nodes_to_coords(G, node_path):
    return [(G.nodes[n]['y'], G.nodes[n]['x']) for n in node_path if 'x' in G.nodes[n] and 'y' in G.nodes[n]]

def _get_trip_shape_coords(trip_id):
    """Get trip shape coordinates from GTFS shapes data"""
    shape_id = trip_to_shape.get(trip_id)
    if shape_id and shape_id in shape_to_points:
        return shape_to_points[shape_id]
    return []

def plot_trip_pair_with_path(start_trip_id, end_trip_id, save_html=None, map_tiles="cartodbpositron"):
    """
    Plot two trips and the walking path between them using GTFS shapes and pathways data.
    """
    # Find pathway row
    row = trip_pathways_df[(trip_pathways_df['start_trip_id'] == start_trip_id) & 
                          (trip_pathways_df['end_trip_id'] == end_trip_id)]
    if row.empty:
        raise ValueError("No pathway found for the given trip pair in trip_pathways_df")
    row = row.iloc[0]
    walking_nodes = row['walking_path_nodes']

    # Get trip shapes
    t1_coords = _get_trip_shape_coords(start_trip_id)
    t2_coords = _get_trip_shape_coords(end_trip_id)
    
    # Get walking path coordinates
    walk_coords = _path_nodes_to_coords(g, walking_nodes)
    
    # Calculate map center from all available coordinates
    all_coords = []
    if t1_coords: all_coords.extend(t1_coords)
    if t2_coords: all_coords.extend(t2_coords)
    if walk_coords: all_coords.extend(walk_coords)
    
    if all_coords:
        center_lat = sum(coord[0] for coord in all_coords) / len(all_coords)
        center_lon = sum(coord[1] for coord in all_coords) / len(all_coords)
    else:
        center_lat, center_lon = 31.2, 29.9  # Default Alexandria center

    m = folium.Map(location=[center_lat, center_lon], zoom_start=14, tiles=map_tiles)

    # Plot trip shapes
    if t1_coords:
        folium.PolyLine(t1_coords, color="#1f77b4", weight=5, opacity=0.8, 
                       tooltip=f"Trip {start_trip_id}").add_to(m)
    if t2_coords:
        folium.PolyLine(t2_coords, color="#ff7f0e", weight=5, opacity=0.8, 
                       tooltip=f"Trip {end_trip_id}").add_to(m)

    # Plot walking path
    if walk_coords:
        folium.PolyLine(walk_coords, color="#2ca02c", weight=6, opacity=0.9, 
                       tooltip=f"Walk {len(walk_coords)} pts").add_to(m)
        # Mark start/end of walking path
        folium.CircleMarker(walk_coords[0], radius=5, color="#2ca02c", fill=True, 
                           tooltip="Walk start").add_to(m)
        folium.CircleMarker(walk_coords[-1], radius=5, color="#2ca02c", fill=True, 
                           tooltip="Walk end").add_to(m)

    if save_html:
        m.save(save_html)
    return m

# Example usage:
# plot_trip_pair_with_path('trip1_id', 'trip2_id', save_html='trip_pair_map.html')


In [13]:
trip_pathways_df[trip_pathways_df['start_trip_id'] == 'oL-cc2-4cc5VXM7r-ShUc-07:00:00']

,start_trip_id,end_trip_id,start_route_id,end_route_id,start_stop_id,end_stop_id,walking_distance_m,walking_path_nodes,start_route_name,end_route_name
110,oL-cc2-4cc5VXM7r-ShUc-07:00:00,0h48FVJYF7q0a-Fr5KoJ2-07:00:00,ujMXATExhhRj9qgxcopKv,2sefONCZ5v5w4bCF1ybYX,402,403,200.395236,"[5720425342, 260758120, 6984479411]",Street 45 - Train Station (El-Shohada Square),Al-Milaha - Train Station (El-Shohada Square)
276,oL-cc2-4cc5VXM7r-ShUc-07:00:00,3JTmhQmnxSU9cPkJSaSlE-07:00:00,ujMXATExhhRj9qgxcopKv,LrIznn0Nk3YYvE6SV4NT_,402,402,0.000000,[5720425342],Street 45 - Train Station (El-Shohada Square),Asafra - Train Station (El-Shohada Square)
495,oL-cc2-4cc5VXM7r-ShUc-07:00:00,4QUtLrt9QqoQBADynOzAM-07:00:00,ujMXATExhhRj9qgxcopKv,SqToUMEw-z1wSe52v0JGh,402,402,0.000000,[5720425342],Street 45 - Train Station (El-Shohada Square),Abu Qir - Train Station (El-Shohada Square)
818,oL-cc2-4cc5VXM7r-ShUc-07:00:00,9iBvMMprqaal91Eesle9O-07:00:00,ujMXATExhhRj9qgxcopKv,2HEouGQcgjyXwlsq69h3S,402,402,0.000000,[5720425342],Street 45 - Train Station (El-Shohada Square),El-Awayed - Train Station (El-Shohada Square)
920,oL-cc2-4cc5VXM7r-ShUc-07:00:00,Bir76FY1196m5b2RCrotL-07:00:00,ujMXATExhhRj9qgxcopKv,eW27jXPyPLkZFOoF5g9lr,402,403,200.395236,"[5720425342, 260758120, 6984479411]",Street 45 - Train Station (El-Shohada Square),Airport - Train Station (El-Shohada Square)
1124,oL-cc2-4cc5VXM7r-ShUc-07:00:00,DQG7BcWUYPP3fo9nY8SBc-07:00:00,ujMXATExhhRj9qgxcopKv,EvMcmidrqDryWEcBEqdU2,402,403,200.395236,"[5720425342, 260758120, 6984479411]",Street 45 - Train Station (El-Shohada Square),El-Mawqaf El-Geded - Train Station (El-Shohada...
1405,oL-cc2-4cc5VXM7r-ShUc-07:00:00,FUo5FExiKwUTpyTUJYA7R-07:00:00,ujMXATExhhRj9qgxcopKv,EqpMMSAPpyyeIs0OPJu7A,402,402,0.000000,[5720425342],Street 45 - Train Station (El-Shohada Square),Asafra - Train Station (El-Shohada Square)
1449,oL-cc2-4cc5VXM7r-ShUc-07:00:00,GK-B6ExLThapIsC2LwCU5-07:00:00,ujMXATExhhRj9qgxcopKv,w5lnA7aOkl9RbW6RzoEdG,402,402,0.000000,[5720425342],Street 45 - Train Station (El-Shohada Square),Hadra - Train Station (El-Shohada Square)
1498,oL-cc2-4cc5VXM7r-ShUc-07:00:00,GVMY3J8hLKwB3k_XzZZNA-07:00:00,ujMXATExhhRj9qgxcopKv,VqsKdXjc_nTkIE8PZUK7V,402,402,0.000000,[5720425342],Street 45 - Train Station (El-Shohada Square),Asafra - Train Station (El-Shohada Square)
1950,oL-cc2-4cc5VXM7r-ShUc-07:00:00,K1fkkoXPyuSsmQKdjzWHd-07:00:00,ujMXATExhhRj9qgxcopKv,QcjXCusjL37XD0vtFPM_z,402,402,0.000000,[5720425342],Street 45 - Train Station (El-Shohada Square),Al-Seyouf (Falaky) - Train Station (El-Shohada...


In [14]:
plot_trip_pair_with_path('oL-cc2-4cc5VXM7r-ShUc-07:00:00', 'aL3IoFC8Ob46ci80-r5J_-07:00:00')